# 4. Category routing, retrieval, and chat API

Inspect category routing and top-$k$ retrieval, then send the same customer question through the FastAPI `/api/v1/chat` endpoint using its in-process test client. Run notebook 3 first to build the index.

In [ ]:
from pathlib import Path
import os

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ingestion").is_dir())
os.chdir(ROOT)

from fastapi.testclient import TestClient
from app.main import app
from app.rag.query_router import detect_category
from app.rag.retriever import retrieve

question = "How can I track my order?"
category = detect_category(question)
hits = retrieve(question, category=category)

print(f"Detected category: {category} | Retrieved chunks: {len(hits)}")
for hit in hits:
    document = hit["document"]
    print({
        "source": document.metadata.get("source"),
        "category": document.metadata.get("category"),
        "score": round(hit["score"], 4),
        "preview": document.page_content[:160].replace("\n", " "),
    })

response = TestClient(app).post("/api/v1/chat", json={"question": question})
response.raise_for_status()
response.json()